<a href="https://colab.research.google.com/github/juliet4b/Single-Cell_RNA-seq_Classification/blob/main/notebooks/SingleCell_Colab_Setup.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


9# Single-Cell RNA-seq — Setup Google Colab

Notebook di ingresso per eseguire il **Project 2** (classificazione single-cell RNA-seq (dataset PBMC))
direttamente in Google Colab.

**Cosa fa questo notebook:**
1. Monta Google Drive e clona il repository (o usa una copia su Drive)
2. Installa le dipendenze da `requirements.txt`
3. Copia i 4 file CSV da Drive in `data_train/` e `data_test/`
4. Esegue la pipeline completa (`FullPipelineTrigger.ipynb`)

**Requisiti:**
- Account Google e accesso al repository GitHub (collaborator, repo privato)
- I 4 CSV nella [cartella Drive condivisa](https://drive.google.com/drive/folders/1k0s5LaNHVm4Bb4TX4rPHkbb1gfz4CacI?usp=sharing) (aggiungere collegamento a *Il mio Drive* prima di Run all)
- Tempo stimato: 15-30 minuti (Colab gratuito)

**Link Colab diretto:** https://colab.research.google.com/github/juliet4b/Single-Cell_RNA-seq_Classification/blob/main/notebooks/SingleCell_Colab_Setup.ipynb

## 1 · Configurazione

In [ ]:
# URL del repository GitHub (privato: serve essere collaborator).
REPO_URL = "https://github.com/juliet4b/Single-Cell_RNA-seq_Classification.git"
REPO_NAME = "Single-Cell_RNA-seq_Classification"

# Cartella su Google Drive con i 4 CSV (X_train, y_train, X_test, y_test).
# Link condiviso: https://drive.google.com/drive/folders/1k0s5LaNHVm4Bb4TX4rPHkbb1gfz4CacI
# In Colab: aprire il link -> "Aggiungi collegamento a Drive" -> la cartella si chiama "data".
DRIVE_CSV_FOLDER = "/content/drive/MyDrive/data"

# Fallback: se git clone fallisce, copia il progetto da questa cartella su Drive.
DRIVE_PROJECT_FOLDER = "/content/drive/MyDrive/SingleCell_project/Single-Cell_RNA-seq_Classification"

# True = esecuzione piu veloce (consigliato su Colab gratuito).
FAST_MODE = True

## 2 · Montare Google Drive

In [ ]:
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

from google.colab import drive

if not os.path.exists("/content/drive"):
    drive.mount("/content/drive")
else:
    print("Drive gia montato.")

## 3 · Clonare il repository

Per un repo **privato** serve essere **collaborator** su GitHub.
Se il clone fallisce (autenticazione), il notebook usa la copia del progetto su Drive.

In [ ]:
os.chdir("/content")

if Path(REPO_NAME).exists():
    print(f"Cartella {REPO_NAME} gia presente — aggiorno con git pull.")
    os.chdir(REPO_NAME)
    subprocess.run(["git", "pull"], check=False)
else:
    print(f"Clone da {REPO_URL} ...")
    r = subprocess.run(["git", "clone", REPO_URL], capture_output=True, text=True)
    if r.returncode != 0:
        print("Clone fallito (repo privato o credenziali mancanti).")
        print("Uso copia da Drive:", DRIVE_PROJECT_FOLDER)
        src = Path(DRIVE_PROJECT_FOLDER)
        if not src.is_dir():
            raise FileNotFoundError(
                f"Clone fallito e cartella Drive non trovata: {src}\n"
                "Soluzioni: (1) accettare invito GitHub e riprovare, "
                "(2) copiare il progetto in Drive al percorso indicato."
            )
        shutil.copytree(src, Path("/content") / REPO_NAME, dirs_exist_ok=True)
        os.chdir(REPO_NAME)
        print("Progetto copiato da Drive.")
    else:
        os.chdir(REPO_NAME)
        print("Clone completato.")

ROOT = Path.cwd().resolve()
print("Directory di lavoro:", ROOT)

## 4 · Installare le dipendenze

In [ ]:
print("Installazione dipendenze...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
print("Dipendenze installate.")

## 5 · Copiare i CSV da Drive

I file CSV non sono nel repository (sono in `.gitignore`).
Vanno copiati manualmente in `data_train/` e `data_test/`.

In [ ]:
csv_map = {
    "X_train.csv": ROOT / "data_train" / "X_train.csv",
    "y_train.csv": ROOT / "data_train" / "y_train.csv",
    "X_test.csv":  ROOT / "data_test"  / "X_test.csv",
    "y_test.csv":  ROOT / "data_test"  / "y_test.csv",
}

(ROOT / "data_train").mkdir(parents=True, exist_ok=True)
(ROOT / "data_test").mkdir(parents=True, exist_ok=True)

src_dir = Path(DRIVE_CSV_FOLDER)
mancanti = []
for nome, dest in csv_map.items():
    sorgente = src_dir / nome
    if not sorgente.is_file():
        mancanti.append(str(sorgente))
    else:
        shutil.copy2(sorgente, dest)
        print(f"OK: {nome} -> {dest.relative_to(ROOT)}")

if mancanti:
    raise FileNotFoundError(
        "CSV mancanti su Drive:\n  " + "\n  ".join(mancanti) +
        f"\nVerificare DRIVE_CSV_FOLDER = {DRIVE_CSV_FOLDER}"
    )
print("\nTutti i CSV copiati correttamente.")

## 6 · Verifica caricamento dati

In [ ]:
sys.path.insert(0, str(ROOT))
from src.data_loader import load_datasets

X_train, X_test, y_train, y_test = load_datasets()
print(f"Train: {X_train.shape}  |  Test: {X_test.shape}")
print(f"Classi: {y_train.nunique()} tipi cellulari")
print("Dati pronti.")

## 7 · Eseguire la pipeline completa

Esegue `notebooks/FullPipelineTrigger.ipynb`, che a sua volta esegue in sequenza:
`data_loader.ipynb` -> `EDA.ipynb` -> `Machine_learning_models.ipynb`.
Gli output vengono salvati dentro ciascun notebook.

In [ ]:
env = os.environ.copy()
env["PYTHONIOENCODING"] = "utf-8"
env["SC_FAST_MODE"] = "1" if FAST_MODE else "0"
timeout_s = 1200 if FAST_MODE else 3600

trigger = ROOT / "notebooks" / "FullPipelineTrigger.ipynb"
cmd = [
    sys.executable, "-m", "jupyter", "nbconvert",
    "--to", "notebook", "--execute", "--inplace",
    f"--ExecutePreprocessor.timeout={timeout_s}",
    str(trigger),
]

print("Avvio pipeline (puo richiedere 15-30 minuti)...")
print("Notebook:", trigger.name)
inizio = time.time()
esito = subprocess.run(cmd, cwd=str(ROOT), env=env, capture_output=True, text=True)
durata = time.time() - inizio

if esito.stdout:
    print(esito.stdout[-3000:] if len(esito.stdout) > 3000 else esito.stdout)
if esito.returncode != 0:
    print("ERRORE pipeline (exit code", esito.returncode, ")")
    if esito.stderr:
        print(esito.stderr[-4000:])
    raise RuntimeError("Pipeline fallita. Controllare l'output sopra.")
else:
    print(f"\nPipeline completata in {durata:.1f} s.")

## 8 · Risultati

La pipeline ha eseguito i notebook in `notebooks/`. Aprire dal pannello file a sinistra:

| Notebook | Contenuto |
|---|---|
| `data_loader.ipynb` | Shape e anteprima dei dati |
| `EDA.ipynb` | Analisi esplorativa completa |
| `Machine_learning_models.ipynb` | Modelli, metriche, feature importance, SHAP |

Ogni notebook contiene gli output inline (grafici, tabelle, metriche).